# Chapter 5: Auto-sklearn — Academic-Grade Automated Pipeline Search

### Why This Matters

**Auto-sklearn** is built directly around the scikit-learn ecosystem you already know from Volume 05, and it distinguishes itself with **meta-learning**: before it even begins searching, it consults a database of results from hundreds of prior OpenML datasets to make an informed initial guess about which algorithms and preprocessing steps are likely to work well on data *similar in shape* to yours — then refines from there with Bayesian optimization, similar in spirit to Optuna's approach in Volume 06.

In [ ]:
import autosklearn.classification

automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=300,     # total search budget, seconds
    per_run_time_limit=60,           # cap on any single model's training time
    seed=42
)
automl.fit(X_train, y_train)

print(automl.leaderboard())
print(automl.sprint_statistics())
predictions = automl.predict(X_test)

`time_left_for_this_task` vs. `per_run_time_limit` is a distinction worth understanding: the first is the *total* budget for the entire search; the second caps how long any *individual* candidate model is allowed to train before being abandoned — protecting the overall search from one unusually slow configuration eating the entire budget.

⚠️ **This is genuinely the most environment-sensitive tool in this volume.** Auto-sklearn has stricter OS and Python-version requirements than the others (notably, no native Windows support at the time of the source material — it requires Linux or WSL/a Linux container on Windows) and a heavier dependency chain. Treat the code above as an accurate reference for when your environment supports it, and don't be discouraged if setting it up takes more troubleshooting than the other four tools in this volume — that's a widely-shared experience, not a sign of doing something wrong. FLAML and AutoGluon are the more forgiving alternatives if you hit installation friction.

### Cheat Sheet — Auto-sklearn

| Task | Code |
|---|---|
| Classifier | `AutoSklearnClassifier(time_left_for_this_task=300, per_run_time_limit=60)` |
| Fit | `automl.fit(X_train, y_train)` |
| Results | `automl.leaderboard()`, `automl.sprint_statistics()` |

---

## Choosing Among the Five: A Decision Guide

| Situation | Reach for |
|---|---|
| You want the fastest path to a first credible model and a leaderboard to eyeball | **PyCaret** |
| You have a hard, small time budget (minutes, on a laptop) | **FLAML** |
| Raw accuracy matters most, and deployment complexity is acceptable | **AutoGluon** |
| You have millions of rows and/or existing H2O infrastructure | **H2O AutoML** |
| You want scikit-learn-native automation with meta-learning, on Linux | **Auto-sklearn** |
| You want full manual control and maximum interpretability | Skip this volume — go back to Volumes 05–06 |

### 🎯 Business Challenge — Volume 07

*Task (from the source material):* run one AutoML library end-to-end on the churn dataset, report the winning model and its ROC-AUC, and write two business recommendations based on its feature importances (or SHAP values, if the tool exposes them).

In [ ]:
# Example using FLAML, chosen here for its light install footprint
from flaml import AutoML

X = pd.get_dummies(df.drop(columns=["churn"]), drop_first=True)
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

automl = AutoML()
automl.fit(X_train, y_train, task="classification", metric="roc_auc", time_budget=90, verbose=0)

from sklearn.metrics import roc_auc_score
proba = automl.predict_proba(X_test)[:, 1]
print("Winning model:", automl.best_estimator)
print("Test ROC-AUC :", round(roc_auc_score(y_test, proba), 4))

# Feature importance, if the winning model exposes it
if hasattr(automl.model.estimator, "feature_importances_"):
    importance = pd.Series(automl.model.estimator.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nTop drivers:\n", importance.head(5))

*Business recommendations (fill in from your actual run's top features):* e.g., "Month-to-month contracts and low tech-support adoption are the two strongest churn drivers identified — prioritize a tech-support upsell campaign specifically targeted at month-to-month customers before their next renewal decision."

---

## Volume 07 — What You Should Be Able to Do Now

- Explain what AutoML automates and, just as importantly, what it does *not* remove the need for (problem framing, leakage checks, business judgment)
- Run PyCaret's `setup → compare_models → tune_model → finalize_model` workflow and interpret its leaderboard
- Choose the right AutoML tool for a given constraint (time budget, dataset size, accuracy priority, deployment simplicity)
- Extract feature importance from an AutoML-selected model and translate it into a business recommendation

**Next: `08_Deep_Learning.md`**